In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


- Perform small-scale fine-tuning experiments
- Observe loss behavior
- Validate hyperparameter sanity
- Gain intuition before full pipeline training

In [ ]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=21d2d38755e6b5b50bdf584a7de09d3a69bedf0f152ddf569e5f95e553a68d8b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)


In [ ]:
dataset = load_dataset("knkarthick/samsum")

small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_val = dataset["validation"].shuffle(seed=42).select(range(100))

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-large")

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [ ]:
def tokenize_fn(batch):
    inputs = tokenizer(
        batch["dialogue"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

    labels = tokenizer(
        text_target=batch["summary"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    labels_ids = []
    for label in labels["input_ids"]:
        labels_ids.append([
            t if t != tokenizer.pad_token_id else -100
            for t in label
        ])

    inputs["labels"] = labels_ids
    return inputs

In [ ]:
train_tok = small_train.map(tokenize_fn, batched=True, remove_columns=small_train.column_names)
val_tok = small_val.map(tokenize_fn, batched=True, remove_columns=small_val.column_names)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./debug_pegasus",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy="steps",
    logging_steps=50,
    save_steps=500,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    report_to="none",
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

Step,Training Loss,Validation Loss
50,3.059100,2.388161
100,2.299500,1.962954
150,2.331900,1.839463
200,1.983100,1.785310
250,2.018700,1.757825
300,1.947000,1.727838
350,1.722400,1.701432
400,1.740900,1.683296
450,1.784300,1.669083
500,1.848500,1.661228


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 256, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=1.8510310974121094, metrics={'train_runtime': 850.3367, 'train_samples_per_second': 1.176, 'train_steps_per_second': 1.176, 'total_flos': 2889464414208000.0, 'train_loss': 1.8510310974121094, 'epoch': 2.0})

In [ ]:
pip install -U transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
model.push_to_hub("hyperchancellor07/pegasus-samsum-dialogue-summarizer")
tokenizer.push_to_hub("hyperchancellor07/pegasus-samsum-dialogue-summarizer")


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pdvscae/model.safetensors:   0%|          | 4.65MB / 2.28GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../tmpbb6vj4hl/spiece.model: 100%|##########| 1.91MB / 1.91MB            

CommitInfo(commit_url='https://huggingface.co/hyperchancellor07/pegasus-samsum-dialogue-summarizer/commit/836e54822631c15a2f5f2c6b997afb5ccf64adcc', commit_message='Upload tokenizer', commit_description='', oid='836e54822631c15a2f5f2c6b997afb5ccf64adcc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hyperchancellor07/pegasus-samsum-dialogue-summarizer', endpoint='https://huggingface.co', repo_type='model', repo_id='hyperchancellor07/pegasus-samsum-dialogue-summarizer'), pr_revision=None, pr_num=None)